# Customer Segmentation with K-Means Clustering
Phân khúc khách hàng dựa trên hành vi mua sắm, thu nhập và tương tác kênh (`dataset/data.csv`).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 1. Load and Inspect Dataset
df = pd.read_csv('dataset/data.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# 2. Select Relevant Behavioral & Spending Features
features = [
    'Income', 'Recency', 'MntTotal', 'MntWines', 'MntMeatProducts',
    'NumWebPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'Age', 'Customer_Days'
]
X = df[features].copy()
print('Missing values check:')
print(X.isnull().sum())
X.describe().round(2)

In [ ]:
# 3. Standardize Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled data shape:', X_scaled.shape)

In [ ]:
# 4. Determine Optimal K via Elbow Method & Silhouette Score
wcss = []
sil_scores = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

for k, w, s in zip(k_range, wcss, sil_scores):
    print(f'K={k} -> WCSS (Inertia): {w:.2f} | Silhouette Score: {s:.4f}')

In [ ]:
# 5. Fit Final K-Means Model with Optimal K=3
optimal_k = 3
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = final_kmeans.fit_predict(X_scaled)
print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# 6. PCA 2D Dimensionality Reduction for Visualization
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(X_scaled)
df['PCA1'] = pca_components[:, 0]
df['PCA2'] = pca_components[:, 1]
explained = pca.explained_variance_ratio_.sum() * 100
print(f'Explained Variance by 2 Components: {explained:.2f}%')

In [ ]:
# 7. Customer Persona Profiling across Clusters
cluster_summary = df.groupby('Cluster')[features].mean().round(2)
print('=== CUSTOMER PERSONA PROFILES (Cluster Means) ===')
print(cluster_summary.T)